# 11 — Kaggle VSL: chuẩn bị MediaPipe 76 điểm cho Pose Transformer

Notebook này **không chạy lại pose extractor**. Nó ưu tiên khôi phục archive canonical đã lưu trên Drive; chỉ gọi Kaggle để lấy một remote ZIP nếu Drive chưa có archive. Top 70 lớp vẫn dùng manifest/test công khai cũ, nhưng dữ liệu train mới giữ đúng thứ tự 76 điểm của uploader và được đóng gói thành một NPZ.

Khi train, mỗi clip được chuyển động thành tensor `[64, 68, 9]`: 25 điểm thân trên + neck + 42 điểm hai tay, với `xyz + velocity xyz + bone xyz`. Không dùng cache `[64,75,7]` cũ vì cache đó hiểu sai thứ tự hai tay và bỏ trục z.

> Dataset công khai không có signer ID. Notebook giữ nguyên test và tách validation có phân tầng từ official train; không trình bày validation này là signer-disjoint.

> Nếu Drive chưa có archive, tạo Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY` hoặc `KAGGLE_API_TOKEN`, rồi bật **Notebook access**.


In [ ]:
#@title Cấu hình
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
KAGGLE_DATASET = 'nguyenanfms/vsl-vietnamese-sign-language-v2'  #@param {type:'string'}
KAGGLE_VERSION = 0  #@param {type:'integer'}
# KAGGLE_VERSION=0 dùng phiên bản public mới nhất, tránh lỗi 404 do version cũ bị gỡ.
DATASET_HANDLE = KAGGLE_DATASET if KAGGLE_VERSION == 0 else f'{KAGGLE_DATASET}/versions/{KAGGLE_VERSION}'
VERSION_TAG = 'latest' if KAGGLE_VERSION == 0 else f'v{KAGGLE_VERSION}'
KAGGLE_KEYPOINT_DIRECTORY = 'processed/processed/keypoints_splited'  #@param {type:'string'}
CLASS_COUNT = 70  #@param {type:'integer'}
MIN_OFFICIAL_TRAIN_SAMPLES = 40  #@param {type:'integer'}
VALIDATION_FRACTION = 0.20  #@param {type:'number'}
SEED = 42  #@param {type:'integer'}
DOWNLOAD_ALL_CANONICAL_KEYPOINTS = True  #@param {type:'boolean'}
SAVE_KEYPOINT_ARCHIVE_TO_DRIVE = True  #@param {type:'boolean'}
COPY_KEYPOINT_FOLDER_TO_DRIVE = False  #@param {type:'boolean'}
OVERWRITE_PACKED_KEYPOINTS = False  #@param {type:'boolean'}

assert CLASS_COUNT > 0
assert MIN_OFFICIAL_TRAIN_SAMPLES >= 2
assert 0 < VALIDATION_FRACTION < 1


In [ ]:
#@title Mount Google Drive và khai báo đường dẫn
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_mediapipe')
SOURCE_SCOPE = 'all_canonical' if DOWNLOAD_ALL_CANONICAL_KEYPOINTS else f'top{CLASS_COUNT}_min{MIN_OFFICIAL_TRAIN_SAMPLES}'
DRIVE_SOURCE_ROOT = DRIVE_ROOT / 'source'
DRIVE_KEYPOINT_ROOT = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}'
DRIVE_KEYPOINT_ARCHIVE = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}_{VERSION_TAG}.tar'
DRIVE_KEYPOINT_ARCHIVE_REPORT = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}_{VERSION_TAG}.json'
SUBSET_ROOT = DRIVE_ROOT / f'subsets/top{CLASS_COUNT}'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
BUILD_REPORT = SUBSET_ROOT / 'manifest_report.json'
PACK_ROOT = SUBSET_ROOT / 'keypoints'
PACKED_KEYPOINTS = PACK_ROOT / 'mediapipe76_front.npz'
PACK_REPORT = PACK_ROOT / 'mediapipe76_front.report.json'
LOCAL_DOWNLOAD_ROOT = Path(f'/content/kaggle-vsl-keypoints-{SOURCE_SCOPE}')
LOCAL_REPO = Path('/content/silent-signal')

for path in (DRIVE_ROOT, DRIVE_SOURCE_ROOT, SUBSET_ROOT, PACK_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


In [ ]:
#@title Khôi phục từ Drive; chỉ dùng Kaggle khi chưa có archive
import hashlib, json, os, shutil, subprocess, sys, tarfile

if not (LOCAL_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
        'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF, f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(LOCAL_REPO)], check=True)
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
print('Commit:', subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip())

def _validate_source_tree(root):
    report_path = root / '_fetch_report.json'
    if not report_path.is_file():
        raise RuntimeError(f'Thiếu fetch report: {report_path}')
    report = json.loads(report_path.read_text(encoding='utf-8'))
    files = sorted(root.rglob('*.npy'))
    expected_version = KAGGLE_VERSION or 'latest'
    expected_strategy = (
        'all_canonical_glosses'
        if DOWNLOAD_ALL_CANONICAL_KEYPOINTS
        else 'descending_official_train_count_then_gloss'
    )
    checks = {
        'status': report.get('status') == 'complete',
        'dataset': report.get('dataset') == KAGGLE_DATASET,
        'version': report.get('requested_version') == expected_version,
        'strategy': report.get('selection_strategy') == expected_strategy,
        'files': report.get('files') == len(files) and len(files) > 0,
    }
    if not DOWNLOAD_ALL_CANONICAL_KEYPOINTS:
        checks['classes'] = report.get('classes') == CLASS_COUNT
        checks['minimum'] = (
            report.get('min_official_train_samples') == MIN_OFFICIAL_TRAIN_SAMPLES
        )
    failed = [name for name, valid in checks.items() if not valid]
    if failed:
        raise RuntimeError(
            f'Source keypoint không hoàn chỉnh ({", ".join(failed)}): {report}'
        )
    return report, files

LOCAL_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE_REPORT = None
local_files = sorted(LOCAL_DOWNLOAD_ROOT.rglob('*.npy'))
if local_files or (LOCAL_DOWNLOAD_ROOT / '_fetch_report.json').exists():
    try:
        LOCAL_SOURCE_REPORT, local_files = _validate_source_tree(LOCAL_DOWNLOAD_ROOT)
        print('Dùng source local đã xác minh:', len(local_files), 'files')
    except (OSError, RuntimeError, ValueError) as error:
        print('Xóa source local dở dang:', error)
        shutil.rmtree(LOCAL_DOWNLOAD_ROOT)
        LOCAL_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
        local_files = []
local_count = len(local_files)
archive_ready = DRIVE_KEYPOINT_ARCHIVE.is_file() and DRIVE_KEYPOINT_ARCHIVE_REPORT.is_file()
if local_count == 0 and archive_ready:
    marker = json.loads(DRIVE_KEYPOINT_ARCHIVE_REPORT.read_text(encoding='utf-8'))
    marker_ok = (
        marker.get('complete') is True
        and marker.get('dataset_handle') == DATASET_HANDLE
        and marker.get('scope') == SOURCE_SCOPE
        and int(marker.get('source_files', 0)) > 0
        and int(marker.get('source_bytes', 0)) > 0
        and (
            marker.get('archive_bytes') is None
            or marker.get('archive_bytes') == DRIVE_KEYPOINT_ARCHIVE.stat().st_size
        )
    )
    if not marker_ok:
        raise RuntimeError(f'Archive marker không khớp dataset: {marker}')
    print('Khôi phục một archive từ Drive:', DRIVE_KEYPOINT_ARCHIVE)
    restore_root = LOCAL_DOWNLOAD_ROOT.with_name(LOCAL_DOWNLOAD_ROOT.name + '-restore')
    if restore_root.exists():
        shutil.rmtree(restore_root)
    restore_root.mkdir(parents=True)
    try:
        with tarfile.open(DRIVE_KEYPOINT_ARCHIVE, mode='r') as archive:
            archive.extractall(restore_root, filter='data')
        LOCAL_SOURCE_REPORT, restored_files = _validate_source_tree(restore_root)
        restored_bytes = sum(path.stat().st_size for path in restored_files)
        restored_report_sha256 = hashlib.sha256(
            (restore_root / '_fetch_report.json').read_bytes()
        ).hexdigest()
        if (
            len(restored_files) != marker['source_files']
            or restored_bytes != marker['source_bytes']
            or (
                marker.get('fetch_report_sha256') is not None
                and restored_report_sha256 != marker['fetch_report_sha256']
            )
        ):
            raise RuntimeError('Archive không khớp file count/byte count/fetch report trong marker.')
        shutil.rmtree(LOCAL_DOWNLOAD_ROOT)
        restore_root.replace(LOCAL_DOWNLOAD_ROOT)
        local_files = restored_files
        local_count = len(local_files)
    except Exception:
        if restore_root.exists():
            shutil.rmtree(restore_root)
        raise

if local_count == 0:
    from google.colab import userdata

    def _read_colab_secret(name):
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''

    kaggle_username = _read_colab_secret('KAGGLE_USERNAME')
    kaggle_key = _read_colab_secret('KAGGLE_KEY')
    kaggle_api_token = _read_colab_secret('KAGGLE_API_TOKEN')
    if kaggle_username and kaggle_key:
        os.environ.pop('KAGGLE_API_TOKEN', None)
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key
        kaggle_auth_mode = 'KAGGLE_USERNAME + KAGGLE_KEY'
    elif kaggle_api_token:
        os.environ['KAGGLE_API_TOKEN'] = kaggle_api_token
        kaggle_auth_mode = 'KAGGLE_API_TOKEN'
    else:
        raise RuntimeError(
            'Drive chưa có archive và notebook thiếu Kaggle Secrets. '
            'Hãy bật KAGGLE_USERNAME + KAGGLE_KEY hoặc KAGGLE_API_TOKEN.'
        )
    print('Kaggle authentication: OK —', kaggle_auth_mode)
    fetch_classes = 0 if DOWNLOAD_ALL_CANONICAL_KEYPOINTS else CLASS_COUNT
    fetch_command = [
        sys.executable, '-u', '-m', 'silent_signal.cli.fetch_kaggle_vsl_mediapipe',
        '--output-root', str(LOCAL_DOWNLOAD_ROOT),
        '--classes', str(fetch_classes),
        '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
        '--workers', '6',
        '--dataset', KAGGLE_DATASET,
        '--version', str(KAGGLE_VERSION),
    ]
    print('+', ' '.join(fetch_command), flush=True)
    subprocess.run(fetch_command, check=True)
    LOCAL_SOURCE_REPORT, local_files = _validate_source_tree(LOCAL_DOWNLOAD_ROOT)
    local_count = len(local_files)

if local_count == 0:
    raise RuntimeError(f'Không tìm thấy .npy trong {LOCAL_DOWNLOAD_ROOT}')
LOCAL_KEYPOINT_ROOT = LOCAL_DOWNLOAD_ROOT
print('Local keypoint root:', LOCAL_KEYPOINT_ROOT)
print('Keypoint files:', f'{local_count:,}')


## Lưu toàn bộ keypoint canonical sang Drive

Mặc định cell dưới đây đóng toàn bộ cây `train/` + `test/` canonical thành **một file `.tar` duy nhất** trên Drive. Cách này nhanh và ổn định hơn việc ghi hàng chục nghìn file nhỏ. Nó không sao chép 72 GB video thô và không lấy `processed_augmented`. Nếu cần xem từng file trực tiếp trên Drive, bật `COPY_KEYPOINT_FOLDER_TO_DRIVE`, nhưng thao tác đó sẽ chậm.

In [ ]:
#@title Đóng gói toàn bộ keypoint thành một file trên Drive
import json, shutil, tarfile, time

LOCAL_SOURCE_REPORT, source_files = _validate_source_tree(LOCAL_KEYPOINT_ROOT)
source_bytes = sum(path.stat().st_size for path in source_files)
fetch_report_path = LOCAL_KEYPOINT_ROOT / '_fetch_report.json'
fetch_report_sha256 = hashlib.sha256(fetch_report_path.read_bytes()).hexdigest()
archive_marker = {
    'dataset_handle': DATASET_HANDLE,
    'source_directory': KAGGLE_KEYPOINT_DIRECTORY,
    'scope': SOURCE_SCOPE,
    'source_files': len(source_files),
    'source_bytes': source_bytes,
    'fetch_report_sha256': fetch_report_sha256,
    'complete': True,
}

if SAVE_KEYPOINT_ARCHIVE_TO_DRIVE:
    archive_ok = False
    if DRIVE_KEYPOINT_ARCHIVE.is_file() and DRIVE_KEYPOINT_ARCHIVE_REPORT.is_file():
        try:
            previous = json.loads(DRIVE_KEYPOINT_ARCHIVE_REPORT.read_text(encoding='utf-8'))
            archive_ok = (
                previous.get('complete') is True
                and previous.get('dataset_handle') == DATASET_HANDLE
                and previous.get('scope') == SOURCE_SCOPE
                and previous.get('source_files') == len(source_files)
                and previous.get('source_bytes') == source_bytes
                and previous.get('fetch_report_sha256') in (None, fetch_report_sha256)
                and previous.get('archive_bytes', DRIVE_KEYPOINT_ARCHIVE.stat().st_size)
                == DRIVE_KEYPOINT_ARCHIVE.stat().st_size
            )
        except (OSError, ValueError):
            archive_ok = False
    if archive_ok:
        archive_marker['archive_bytes'] = DRIVE_KEYPOINT_ARCHIVE.stat().st_size
        DRIVE_KEYPOINT_ARCHIVE_REPORT.write_text(
            json.dumps(archive_marker, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print('Archive đã hoàn chỉnh, bỏ qua:', DRIVE_KEYPOINT_ARCHIVE)
    else:
        partial = DRIVE_KEYPOINT_ARCHIVE.with_suffix(DRIVE_KEYPOINT_ARCHIVE.suffix + '.partial')
        if partial.exists():
            partial.unlink()
        started = time.perf_counter()
        with tarfile.open(partial, mode='w') as archive:
            for position, source in enumerate(source_files, start=1):
                archive.add(source, arcname=source.relative_to(LOCAL_KEYPOINT_ROOT), recursive=False)
                if position == 1 or position == len(source_files) or position % 1000 == 0:
                    elapsed = time.perf_counter() - started
                    rate = position / elapsed if elapsed else 0
                    eta = (len(source_files) - position) / rate / 60 if rate else 0
                    print(f'archive {position:,}/{len(source_files):,} | ETA={eta:.1f} min')
            fetch_report = LOCAL_KEYPOINT_ROOT / '_fetch_report.json'
            if fetch_report.is_file():
                archive.add(fetch_report, arcname='_fetch_report.json', recursive=False)
        partial.replace(DRIVE_KEYPOINT_ARCHIVE)
        archive_marker['archive_bytes'] = DRIVE_KEYPOINT_ARCHIVE.stat().st_size
        archive_marker['elapsed_seconds'] = round(time.perf_counter() - started, 1)
        DRIVE_KEYPOINT_ARCHIVE_REPORT.write_text(
            json.dumps(archive_marker, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print('Đã lưu một archive trên Drive:', DRIVE_KEYPOINT_ARCHIVE)
        print(json.dumps(archive_marker, ensure_ascii=False, indent=2))
else:
    print('Bỏ qua tạo archive theo cấu hình.')

if COPY_KEYPOINT_FOLDER_TO_DRIVE:
    copied = skipped = 0
    copied_bytes = 0
    started = time.perf_counter()
    for position, source in enumerate(source_files, start=1):
        relative = source.relative_to(LOCAL_KEYPOINT_ROOT)
        destination = DRIVE_KEYPOINT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size == source.stat().st_size:
            skipped += 1
        else:
            shutil.copy2(source, destination)
            copied += 1
            copied_bytes += source.stat().st_size
        if position == 1 or position == len(source_files) or position % 500 == 0:
            elapsed = time.perf_counter() - started
            rate = position / elapsed if elapsed else 0
            eta = (len(source_files) - position) / rate / 60 if rate else 0
            print(f'{position:,}/{len(source_files):,} | copied={copied:,} | skipped={skipped:,} | ETA={eta:.1f} min')
    persisted = sum(1 for _ in DRIVE_KEYPOINT_ROOT.rglob('*.npy'))
    marker = {
        'dataset_handle': DATASET_HANDLE,
        'source_directory': KAGGLE_KEYPOINT_DIRECTORY,
        'selection': SOURCE_SCOPE,
        'source_files': len(source_files),
        'persisted_files': persisted,
        'complete': persisted == len(source_files),
    }
    (DRIVE_KEYPOINT_ROOT / '_copy_report.json').write_text(
        json.dumps(marker, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    if not marker['complete']:
        raise RuntimeError(f'Copy chưa đủ: {marker}')
    print(json.dumps(marker, ensure_ascii=False, indent=2))
else:
    print('Không copy từng file nhỏ; dùng archive .tar ở trên.')


In [ ]:
#@title Xác nhận source code đã cài
assert (LOCAL_REPO / '.git').is_dir()
print('Commit:', subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
#@title Chọn lớp, giữ test và tạo validation
import importlib, sys
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import silent_signal.cli.prepare_kaggle_vsl_mediapipe as prepare_mediapipe_module
prepare_mediapipe_module = importlib.reload(prepare_mediapipe_module)
prepare_mediapipe_main = prepare_mediapipe_module.main
print('Preparation module:', prepare_mediapipe_module.__file__)

# Quét toàn bộ bản local; lệnh build bên dưới chỉ chọn top 70 lớp để train.
KEYPOINT_ROOT_FOR_CONVERSION = LOCAL_KEYPOINT_ROOT
arguments = [
    'build',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--labels', str(LABELS),
    '--selection', str(SELECTION),
    '--report', str(BUILD_REPORT),
    '--classes', str(CLASS_COUNT),
    '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
    '--validation-fraction', str(VALIDATION_FRACTION),
    '--seed', str(SEED),
    '--dataset-handle', DATASET_HANDLE,
]
print('+ ss-prepare-kaggle-vsl-mediapipe', ' '.join(arguments), flush=True)
return_code = prepare_mediapipe_main(arguments)
if return_code:
    raise RuntimeError(f'Manifest build thất bại với mã {return_code}.')
missing_outputs = [path for path in (MANIFEST, LABELS, SELECTION, BUILD_REPORT) if not path.is_file()]
if missing_outputs:
    raise RuntimeError(f'Lệnh build kết thúc nhưng thiếu output: {missing_outputs}')
print('Manifest build: OK —', MANIFEST)


In [ ]:
#@title Đóng gói đúng các clip top-N thành một NPZ trên Drive — có resume
import importlib, sys
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import silent_signal.cli.prepare_kaggle_vsl_mediapipe as prepare_mediapipe_module
prepare_mediapipe_module = importlib.reload(prepare_mediapipe_module)
prepare_mediapipe_main = prepare_mediapipe_module.main

arguments = [
    'pack',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--output', str(PACKED_KEYPOINTS),
    '--report', str(PACK_REPORT),
    '--dataset-handle', DATASET_HANDLE,
    '--progress-every', '100',
]
if OVERWRITE_PACKED_KEYPOINTS:
    arguments.append('--overwrite')
print('+ ss-prepare-kaggle-vsl-mediapipe', ' '.join(arguments), flush=True)
return_code = prepare_mediapipe_main(arguments)
if return_code:
    raise RuntimeError(f'Đóng gói keypoint thất bại với mã {return_code}.')
for required in (PACKED_KEYPOINTS, PACK_REPORT):
    if not required.is_file():
        raise RuntimeError(f'Lệnh pack kết thúc nhưng thiếu output: {required}')
print('Packed keypoints: OK —', PACKED_KEYPOINTS)


In [ ]:
#@title Kiểm tra layout mới và tensor 68 điểm × 9 đặc trưng
import json
from silent_signal.data.keypoint_pack import read_packed_keypoints
from silent_signal.preprocessing.mediapipe_features import (
    MediaPipeFeatureConfig,
    mediapipe_graph_features,
)

packed = read_packed_keypoints(PACKED_KEYPOINTS)
config = MediaPipeFeatureConfig(target_frames=64)
features, joint_mask, frame_mask = mediapipe_graph_features(packed.sequence(0), config)
assert features.shape == (64, 68, 9), features.shape
assert joint_mask.shape == (64, 68), joint_mask.shape
assert frame_mask.shape == (64,), frame_mask.shape
print('Samples:', len(packed.sample_ids))
print('First sample:', packed.sample_ids[0])
print('Features:', features.shape, '| joint mask:', joint_mask.shape)
print('Metadata:', json.dumps(packed.metadata, ensure_ascii=False, indent=2))


In [ ]:
#@title Kết quả cuối
import json

pack_report = json.loads(PACK_REPORT.read_text(encoding='utf-8'))
build_report = json.loads(BUILD_REPORT.read_text(encoding='utf-8'))
print(json.dumps({
    'classes': build_report['classes'],
    'clips': build_report['clips'],
    'splits': build_report['splits'],
    'packed_samples': pack_report['samples'],
    'packed_frames': pack_report['frames'],
    'packed_gib': round(pack_report['output_bytes'] / 1024**3, 3),
}, ensure_ascii=False, indent=2))
print('Full canonical archive:', DRIVE_KEYPOINT_ARCHIVE)
print('Packed top-N keypoints:', PACKED_KEYPOINTS)
print('Manifest:', MANIFEST)
print('Labels:', LABELS)


## Đầu ra

- `source/keypoints_splited_all_canonical_latest.tar`: toàn bộ file canonical `[T,76,3]`; lần chạy sau được khôi phục từ Drive, không cần gọi lại Kaggle.
- `subsets/top70/manifest.csv`: nhãn liên tục và split hiện tại; test công khai được giữ nguyên.
- `subsets/top70/keypoints/mediapipe76_front.npz`: chỉ các clip top 70 đã chọn, vẫn giữ raw 76 điểm để tạo augmentation khác nhau ở mỗi epoch.
- Khi train, raw sequence được ánh xạ đúng sang `[64,68,9]`.

Notebook 12 chỉ cần NPZ, manifest và labels trên Drive. Graph cache `[64,75,7]` cũ được giữ lại nếu đã tồn tại nhưng không còn được sử dụng.
